# Air Quality and Temperature Analysis
**Author:** [kashevchenko](https://github.com/kashevchenko)

## 1. Research introduction
Is temperature associated with pollutant concentration? For each pollutant, the primary null is zero population Pearson correlation (no linear association); the two-sided alternative is a nonzero correlation. This is an association study, not a causal experiment. Statistical calculations below reuse the tested module in `src/analysis.py`.

## 2. Dataset description and portable setup
The supplied dataset has 500 rows and ten numeric variables. Source, geography, period, sampling design, physical units and measured versus simulated status are unknown. Raw data redistribution has not been authorized, so upload your copy when prompted in Colab. Locally, put it in `data/air_quality_dataset.csv` or set `AIR_QUALITY_DATA`.

The setup finds a local repository first. In Colab it clones the repository when needed and installs its pinned dependencies. No credentials are required for the public repository.

In [1]:
from pathlib import Path
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "air-quality-temperature-analysis"]
ROOT = next((p for p in candidates if (p / "src/analysis.py").exists()), None)
if ROOT is None and IN_COLAB:
    ROOT = Path.cwd() / "air-quality-temperature-analysis"
    subprocess.run(["git", "clone", "https://github.com/kashevchenko/air-quality-temperature-analysis.git", str(ROOT)], check=True)
if ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks directory.")
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import display, Image
from analysis import load_and_clean, correlations, regressions, evaluate_models, figures, run, COLUMNS
DATA = Path(os.environ.get("AIR_QUALITY_DATA", str(ROOT / "data/air_quality_dataset.csv")))
if not DATA.exists() and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    csvs = [name for name in uploaded if name.lower().endswith(".csv")]
    if len(csvs) != 1:
        raise ValueError("Upload exactly one authorized CSV file.")
    DATA = Path(csvs[0]).resolve()
raw, clean, audit = load_and_clean(DATA)
print("Original shape:", raw.shape)
display(raw.dtypes.astype(str).rename("dtype").to_frame())

Original shape: (500, 10)


,dtype
PM2.5,float64
PM10,float64
NOx,float64
NO2,float64
SO2,float64
VOCs,float64
CO,float64
CO2,float64
CH4,float64
Temperature,float64


## 3. Data cleaning
Original bytes are never changed. Non-numeric values are coerced only in a working copy and audited. Complete cases across ten variables are used; rows with nonfinite values or negative pollutant concentrations are excluded. Negative readings may have an instrument-related meaning that is unknown here, so a sensitivity analysis retains them. Duplicate analysis rows are removed. Values beyond 1.5 IQR are flagged but retained. This policy may introduce selection bias.

In [2]:
import json
print(json.dumps(audit, indent=2))

{
  "input_sha256": "aaf87134601406a882713a37e990b466945e680d9dd3ee00161ecf62b1ba1f97",
  "raw_shape": [
    500,
    10
  ],
  "raw_dtypes": {
    "PM2.5": "float64",
    "PM10": "float64",
    "NOx": "float64",
    "NO2": "float64",
    "SO2": "float64",
    "VOCs": "float64",
    "CO": "float64",
    "CO2": "float64",
    "CH4": "float64",
    "Temperature": "float64"
  },
  "missing_by_column": {
    "PM2.5": 0,
    "PM10": 0,
    "NOx": 0,
    "NO2": 0,
    "SO2": 0,
    "VOCs": 0,
    "CO": 0,
    "CO2": 0,
    "CH4": 0,
    "Temperature": 0
  },
  "coercion_failures": {
    "Temperature": 0,
    "PM2.5": 0,
    "PM10": 0,
    "NOx": 0,
    "NO2": 0,
    "SO2": 0,
    "VOCs": 0,
    "CO": 0,
    "CO2": 0,
    "CH4": 0
  },
  "nonfinite_or_missing_by_column": {
    "Temperature": 0,
    "PM2.5": 0,
    "PM10": 0,
    "NOx": 0,
    "NO2": 0,
    "SO2": 0,
    "VOCs": 0,
    "CO": 0,
    "CO2": 0,
    "CH4": 0
  },
  "raw_duplicate_records": 0,
  "negative_by_pollutant": {
    "PM2.

## 4. Exploratory Data Analysis
Standard deviation uses the sample convention. Quartiles are empirical 25th, 50th (median), and 75th percentiles. Axes use recorded values without invented units.

In [3]:
display(clean.describe().T.rename(columns={"50%": "median"}))

,count,mean,std,min,25%,median,75%,max
Temperature,496.0,30.465969,2.959749,19.581349,28.502023,30.448228,32.628851,39.778713
PM2.5,496.0,54.773527,22.831536,10.757597,36.481666,48.038599,73.944487,127.616632
PM10,496.0,84.869644,30.611655,12.932337,61.118607,78.352268,110.634069,171.594187
NOx,496.0,92.040600,54.810039,3.509005,35.457263,95.452284,134.733873,225.886935
NO2,496.0,42.578065,26.345890,0.838370,17.448632,39.435826,64.109563,115.916428
SO2,496.0,11.567660,7.291741,1.241808,5.984712,8.716235,17.198766,31.941860
VOCs,496.0,112.447214,39.516995,27.555617,81.821262,105.480626,140.723792,231.926017
CO,496.0,2.184144,0.824636,0.366992,1.543188,2.050378,2.731118,4.798935
CO2,496.0,439.584359,51.298302,366.063158,397.492771,421.160157,491.526292,558.924256
CH4,496.0,2.131707,0.392516,1.388407,1.837395,2.068412,2.386709,3.352266


## 5. Correlation analysis
Pearson measures linear association; Spearman measures monotonic rank association. P-values are two-sided. Benjamini–Hochberg correction is applied separately to the nine primary Pearson tests and nine secondary Spearman tests. The heatmap is descriptive; its other cells are not additional hypothesis claims.

In [4]:
corr = correlations(clean)
display(corr.sort_values("pearson_r", key=lambda x: x.abs(), ascending=False))

,pollutant,n,pearson_r,pearson_p,spearman_r,spearman_p,pearson_q,pearson_significant,spearman_q,spearman_significant
7,CO2,496,0.596820,3.466973e-49,0.602470,2.516880e-50,3.120275e-48,True,2.265192e-49,True
2,NOx,496,0.540824,5.053976e-39,0.558945,4.211965e-42,2.274289e-38,True,1.263589e-41,True
3,NO2,496,0.535722,3.445395e-38,0.566323,2.068613e-43,1.033618e-37,True,9.308758e-43,True
4,SO2,496,0.500564,8.168543e-33,0.483146,2.250374e-30,1.837922e-32,True,4.050672e-30,True
6,CO,496,0.449654,4.655265e-26,0.491371,1.650040e-31,8.379477e-26,True,3.712591e-31,True
0,PM2.5,496,0.444757,1.819038e-25,0.449346,5.075230e-26,2.728557e-25,True,7.612846e-26,True
1,PM10,496,0.420757,1.061369e-22,0.428990,1.264312e-23,1.364617e-22,True,1.625544e-23,True
8,CH4,496,0.413023,7.429285e-22,0.427547,1.843142e-23,8.357946e-22,True,2.073535e-23,True
5,VOCs,496,0.386310,4.226797e-19,0.406878,3.365435e-21,4.226797e-19,True,3.365435e-21,True


## 6. Statistical hypothesis testing
Use adjusted p-values (q) below 0.05 to reject a null within its test family. Failure to reject does not establish that the relationship is absent. These calculations assume independent observations; this cannot be checked without timestamps or locations. Pearson's usual p-value additionally relies on distributional assumptions; Spearman is a complementary analysis, not a cure for dependence.

In [5]:
print("Pearson rejections:", int(corr.pearson_significant.sum()), "of", len(corr))
print("Spearman rejections:", int(corr.spearman_significant.sum()), "of", len(corr))

Pearson rejections: 9 of 9
Spearman rejections: 9 of 9


## 7. Linear regression
For each pollutant, fit $Y=\beta_0+\beta_1 T+\epsilon$ using all cleaned observations for inference. HC3 heteroskedasticity-robust standard errors and t-based 95% confidence intervals are used throughout. Intervals are individual, not simultaneous. An additional BH correction covers nine robust slope tests. HC3 does not fix temporal or spatial dependence, confounding or a wrong functional form. Full-sample R² is explanatory fit, not held-out prediction performance.

In [6]:
regression = regressions(clean)
display(regression)

,pollutant,term,coefficient,robust_se,p_value,ci_low,ci_high,in_sample_r2,n,covariance,slope_q
0,PM2.5,const,-49.751065,10.061996,1.047999e-06,-69.520650,-29.981480,0.197809,496,HC3,NaN
1,PM2.5,Temperature,3.430864,0.338126,4.220100e-22,2.766521,4.095207,0.197809,496,HC3,5.425843e-22
2,PM10,const,-47.710408,11.445816,3.622662e-05,-70.198893,-25.221923,0.177036,496,HC3,NaN
3,PM10,Temperature,4.351742,0.382850,9.344055e-27,3.599527,5.103958,0.177036,496,HC3,1.401608e-26
4,NOx,const,-213.083056,22.437888,9.300730e-20,-257.168518,-168.997594,0.292490,496,HC3,NaN
5,NOx,Temperature,10.015229,0.753753,1.160137e-34,8.534273,11.496185,0.292490,496,HC3,3.480412e-34
6,NO2,const,-102.704057,10.100713,3.522255e-22,-122.549713,-82.858401,0.286998,496,HC3,NaN
7,NO2,Temperature,4.768669,0.341792,1.551554e-37,4.097124,5.440214,0.286998,496,HC3,6.981992e-37
8,SO2,const,-26.003216,3.114406,6.954494e-16,-32.122333,-19.884100,0.250565,496,HC3,NaN
9,SO2,Temperature,1.233208,0.105054,3.088869e-28,1.026801,1.439615,0.250565,496,HC3,5.559964e-28


## 8. Polynomial regression
Compare the prespecified degree-two model $Y=\beta_0+\beta_1T+\beta_2T^2+\epsilon$ with linear regression and a training-mean baseline. Split cleaned rows once, 80% training and 20% testing, seed 42. All pollutants and models share the split. Scaling and fitting use training rows only; coefficients below are converted back to the original temperature scale. No test-set tuning is performed.

In [7]:
comparison, models, split, coefficients = evaluate_models(clean)
display(coefficients)
display(split.partition.value_counts())

,pollutant,model,beta0,beta1,beta2
0,PM2.5,Linear,-49.711878,3.438477,0.000000
1,PM2.5,Polynomial (degree 2),169.231358,-11.319786,0.246103
2,PM10,Linear,-38.265765,4.061975,0.000000
3,PM10,Polynomial (degree 2),203.417739,-12.229137,0.271665
4,NOx,Linear,-204.022451,9.686243,0.000000
5,NOx,Polynomial (degree 2),-358.349966,20.088968,-0.173472
6,NO2,Linear,-100.832728,4.695929,0.000000
7,NO2,Polynomial (degree 2),-130.031209,6.664106,-0.032821
8,SO2,Linear,-25.222992,1.210195,0.000000
9,SO2,Polynomial (degree 2),58.760882,-4.450889,0.094402


partition
train    396
test     100
Name: count, dtype: int64

## 9. Model evaluation and visualizations
Evaluate R², RMSE and MAE on held-out rows only. A negative test R² means the model performs worse than the test-mean reference used by that metric. The separate mean baseline predicts the training mean. Compare errors within a pollutant, as units and scales differ. A single split provides no uncertainty estimate for model rankings and is not a forecast validation.

In [8]:
display(comparison)
OUT = ROOT / "results"
# Regenerate every reproducible artifact, including the sensitivity table.
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    result = run(DATA, OUT)
for name in ["distributions", "correlation_matrix", "temperature_correlations", "regression_analysis", "polynomial_comparison", "model_comparison"]:
    display(Image(filename=str(OUT / f"{name}.png"), width=900))

,pollutant,model,n_train,n_test,test_r2,test_rmse,test_mae
0,PM2.5,Mean baseline,396,100,-0.000244,21.742223,17.719810
1,PM2.5,Linear,396,100,0.166215,19.850802,15.508936
2,PM2.5,Polynomial (degree 2),396,100,0.133773,20.233308,15.454195
3,PM10,Mean baseline,396,100,-0.003677,30.912193,27.517539
4,PM10,Linear,396,100,0.221444,27.225615,22.878773
5,PM10,Polynomial (degree 2),396,100,0.228598,27.100251,22.498267
6,NOx,Mean baseline,396,100,-0.019948,54.375250,47.388352
7,NOx,Linear,396,100,0.300333,45.035801,37.295300
8,NOx,Polynomial (degree 2),396,100,0.305763,44.860697,37.129583
9,NO2,Mean baseline,396,100,-0.012551,27.689064,23.225571


## 10. Research conclusions
The following summary is computed from the loaded file, rather than hardcoded. A model with a slightly lower error on this split is not necessarily superior in the population. Unknown provenance and independence prevent generalization to specific environments. Correlation does not imply causation.

In [9]:
strongest = corr.loc[corr.pearson_r.abs().idxmax()]
print(f"Strongest absolute Pearson association: {strongest.pollutant}; r={strongest.pearson_r:.4f}, q={strongest.pearson_q:.4g}.")
pivot = comparison.pivot(index="pollutant", columns="model", values="test_rmse")
print("Quadratic has lower test RMSE than linear for", int((pivot["Polynomial (degree 2)"] < pivot["Linear"]).sum()), "of 9 pollutants.")
print("Eligible records:", len(clean), "of", len(raw))
display(pd.read_csv(OUT / "correlation_sensitivity_keep_negatives.csv"))

Strongest absolute Pearson association: CO2; r=0.5968, q=3.12e-48.
Quadratic has lower test RMSE than linear for 4 of 9 pollutants.
Eligible records: 496 of 500


,pollutant,n,pearson_r,pearson_p,spearman_r,spearman_p,pearson_q,pearson_significant,spearman_q,spearman_significant
0,PM2.5,500,0.443328,1.734191e-25,0.446005,8.232775e-26,2.601286e-25,True,1.234916e-25,True
1,PM10,500,0.423699,3.349587e-23,0.430756,5.251929e-24,4.306612e-23,True,6.752480e-24,True
2,NOx,500,0.542993,1.095463e-39,0.561303,7.575959e-43,4.929583e-39,True,2.272788e-42,True
3,NO2,500,0.537200,9.993015e-39,0.568053,4.612556e-44,2.997904e-38,True,2.075650e-43,True
4,SO2,500,0.500631,4.468390e-33,0.481636,2.119529e-30,1.005388e-32,True,3.815152e-30,True
5,VOCs,500,0.389045,1.625750e-19,0.408929,1.408857e-21,1.625750e-19,True,1.408857e-21,True
6,CO,500,0.453265,1.056319e-26,0.495702,2.295121e-32,1.901375e-26,True,5.164022e-32,True
7,CO2,500,0.598649,6.115936e-50,0.604509,3.867516e-51,5.504343e-49,True,3.480764e-50,True
8,CH4,500,0.413543,4.469204e-22,0.426793,1.494590e-23,5.027854e-22,True,1.681414e-23,True
